# Day-Ahead Benchmark Comparison

This notebook runs two local benchmark scenarios for the electric bus fleet backend:

- `dumb_charging_no_v2g`
- `optimization_no_v2g`

It then builds a comparison workbook with `day_ahead_summary`, `agent_reasoning`, `day_ahead_plan`, and `scenario_comparison` sheets.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from openpyxl import load_workbook
from openpyxl.styles import Font
import pandas as pd

APP_DIR = Path.cwd()
INPUT_FILE = APP_DIR / 'Files' / 'Inputs.xlsx'
OUTPUT_WORKBOOK = APP_DIR / 'Files' / 'day_ahead_local_comparison.xlsx'
DUMB_JSON = APP_DIR / 'Files' / 'dumb_charging_result.json'
SMART_JSON = APP_DIR / 'Files' / 'no_v2g_optimization_result.json'
OUTPUT_WORKBOOK


In [ ]:
commands = [
    [sys.executable, 'run_dumb_charging.py', '--input', str(INPUT_FILE), '--output', str(DUMB_JSON), '--summary-workbook', str(OUTPUT_WORKBOOK)],
    [sys.executable, 'run_no_v2g_optimization.py', '--input', str(INPUT_FILE), '--output', str(SMART_JSON), '--summary-workbook', str(OUTPUT_WORKBOOK)],
]

for command in commands:
    print('Running:', ' '.join(command))
    subprocess.run(command, cwd=APP_DIR, check=True)

print('Scenario runs completed.')


In [ ]:
with DUMB_JSON.open() as f:
    dumb = json.load(f)
with SMART_JSON.open() as f:
    smart = json.load(f)

wb = load_workbook(OUTPUT_WORKBOOK)
summary_ws = wb['day_ahead_summary']
summary_headers = [summary_ws.cell(row=1, column=i).value for i in range(1, summary_ws.max_column + 1)]
summary_rows = []
for row_idx in range(2, summary_ws.max_row + 1):
    row = {summary_headers[i-1]: summary_ws.cell(row=row_idx, column=i).value for i in range(1, summary_ws.max_column + 1)}
    summary_rows.append(row)
summary_by_scenario = {row['scenario']: row for row in summary_rows[-2:]}

if 'day_ahead_plan' in wb.sheetnames:
    del wb['day_ahead_plan']
plan_ws = wb.create_sheet('day_ahead_plan')
plan_headers = ['scenario', 'run_timestamp_utc', 'timestep', 'w_buy', 'w_sell'] + [item for i in range(1, 9) for item in (f'bus_{i}_power_kw', f'bus_{i}_kwh')] + ['buy_price', 'sell_price']
plan_ws.append(plan_headers)
for cell in plan_ws[1]:
    cell.font = Font(bold=True)

for result in [dumb, smart]:
    scenario = result['scenario']
    run_timestamp = summary_by_scenario[scenario]['run_timestamp_utc']
    for t in range(len(result['w_buy'])):
        row = [scenario, run_timestamp, t, result['w_buy'][t], result['w_sell'][t]]
        for b in range(8):
            row.append(result['power_kw'][b][t] if b < len(result.get('power_kw', [])) else None)
            row.append(result['energy'][b][t] if b < len(result['energy']) else None)
        row.append(result['S_buy'][t] if t < len(result['S_buy']) else None)
        row.append(result['S_sell'][t] if t < len(result['S_sell']) else None)
        plan_ws.append(row)

if 'scenario_comparison' in wb.sheetnames:
    del wb['scenario_comparison']
comp_ws = wb.create_sheet('scenario_comparison', 0)
comp_ws['A1'] = 'Metric'
comp_ws['B1'] = 'dumb_charging_no_v2g'
comp_ws['C1'] = 'optimization_no_v2g'
for cell in comp_ws[1]:
    cell.font = Font(bold=True)

metrics = [
    'pto_daily_cost', 'net_daily_cost', 'total_kwh_bought', 'total_kwh_sold',
    'peak_grid_import_kw', 'peak_grid_export_kw', 'cost_per_kwh_served',
    'minimum_soc_before_departure', 'average_soc_before_departure',
    'number_of_risked_departures', 'trip_feasibility_rate', 'end_of_day_average_soc',
    'share_of_charging_in_low_price_hours', 'share_of_charging_in_high_price_hours',
    'battery_throughput_kwh', 'delta_vs_dumb_cost', 'delta_vs_dumb_cost_pct',
]
for idx, metric in enumerate(metrics, start=2):
    comp_ws.cell(row=idx, column=1, value=metric)
    comp_ws.cell(row=idx, column=2, value=summary_by_scenario['dumb_charging_no_v2g'].get(metric))
    comp_ws.cell(row=idx, column=3, value=summary_by_scenario['optimization_no_v2g'].get(metric))

for ws in [comp_ws, plan_ws, wb['day_ahead_summary'], wb['agent_reasoning']]:
    for col in ws.columns:
        max_len = 0
        col_letter = col[0].column_letter
        for cell in col[:100]:
            if cell.value is None:
                continue
            max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 40)

wb.save(OUTPUT_WORKBOOK)
print('Workbook saved to', OUTPUT_WORKBOOK)


In [ ]:
summary_df = pd.read_excel(OUTPUT_WORKBOOK, sheet_name='day_ahead_summary')
comparison_df = pd.read_excel(OUTPUT_WORKBOOK, sheet_name='scenario_comparison')
display(summary_df.tail(2))
display(comparison_df)
